# Chasing the Aha Moment — saved-adapter evaluation
**No training.** Compare the adapter from your completed four-response pilot with its original base model. Both use identical prompts, greedy decoding, 512-token limits and a frozen scoring policy. Evaluate **32 new development questions**, excluding the previous development examples and all saved training examples. The official GSM8K test split remains untouched.

## Instructions
1. Open in a fresh Colab T4 session. Run installation, then **Runtime → Restart session**.
2. Skip installation after restarting. Run all remaining cells in order.
3. The run path below is filled from your screenshot: `20261025T191033502154Z`. If it cannot find that completed run, stop and send the error rather than rerunning training.
4. The evaluation cell saves every answer and prints progress. It pauses after approximately 20 minutes (finishing the current question pair first). If it says **PAUSED**, rerun only that evaluation cell; completed answers are skipped. After a disconnection, restart the notebook from Drive mounting using the same file and settings.
5. Run the final summary cell. Send `evaluation_summary.json` and `evaluation_predictions.jsonl` only once the summary says `complete: true`.

Expect roughly 20–35 minutes of generation based on your earlier timings, plus model loading; this is an estimate, not a guarantee. CPU tests and syntax checks passed; this evaluation-only GPU workflow has not been run here.


In [ ]:
%pip install "transformers==4.56.2" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.1.1" "bitsandbytes==0.48.1"
# Restart session once now. Do not rerun this cell after restarting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, time, hashlib, subprocess, importlib.metadata
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime first.'
RUN = Path('/content/drive/MyDrive/aha_pilot_v4/20261025T191033502154Z')
required=['config.json','data_manifest.json','model_revision.json','train.jsonl','dev.jsonl',
          'training.json','adapter/adapter_config.json','adapter/adapter_model.safetensors']
missing=[name for name in required if not (RUN/name).is_file()]
assert not missing, f'Missing files in {RUN}: {missing}. Stop and send this message; do not retrain.'
CFG=json.loads((RUN/'config.json').read_text())
manifest=json.loads((RUN/'data_manifest.json').read_text())
revision=json.loads((RUN/'model_revision.json').read_text())
training=json.loads((RUN/'training.json').read_text())
assert training['completed_steps']==20, 'Expected the completed 20-step pilot.'
assert CFG['model']=='Qwen/Qwen2.5-1.5B-Instruct'
assert revision.get('resolved_commit'), 'Saved base-model revision is missing; stop before loading.'
print('Evaluating saved adapter:',RUN/'adapter')
print('GPU:',torch.cuda.get_device_name(0))
N_QUESTIONS=32
MAX_NEW_TOKENS=512
SESSION_BUDGET_SECONDS=1200


In [ ]:
from pathlib import Path
Path('/content/eval_scoring.py').write_text('"""Frozen v3 evaluation policy: unique tagged numeric answer + bounded presentation normalization.\nNo searching for reference numbers or inferring answers from intermediate work.\nUntagged responses remain unparseable. Units are labels; no unit conversion.\n"""\nimport re\nfrom decimal import Decimal\nVERSION = \'v3-tagged-numeric-presentation\'\nNUM = r\'[+-]?(?:\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.\\d+)?\'\nUNITS = r\'(?:%|percent|hours?|minutes?|seconds?|days?|weeks?|months?|years?|dollars?|cents?|pounds?|ounces?|miles?|feet|foot|inches|inch|square feet|square foot|pages?|books?|posters?|people|persons?|cows?|pigs?|fish|stamps?|pieces?|bottles?)\'\nVALUE = re.compile(rf\'\\s*[$£€]?\\s*({NUM})\\s*(?:{UNITS})?\\s*[.!]?\\s*\', re.I)\nTAIL = re.compile(rf\'\\s*(?:{UNITS})?\\s*[.!]?\\s*\', re.I)\nFORMAT = re.compile(r\'\\s*<reasoning>(.*?)</reasoning>\\s*<answer>(.*?)</answer>\\s*\', re.S)\ndef numeric(value):\n    value = str(value).strip().strip(\'*_\').strip()\n    m = VALUE.fullmatch(value)\n    return Decimal(m[1].replace(\',\', \'\')) if m else None\n\ndef extract(text):\n    if text.count(\'<answer>\') != 1 or text.count(\'</answer>\') != 1:\n        return None, \'missing_or_duplicate_answer_tags\'\n    m = re.search(r\'<answer>(.*?)</answer>\', text, re.S)\n    if not m:\n        return None, \'invalid_tag_order\'\n    tail = text[m.end():].strip().strip(\'*_\').strip()\n    # Remove emphasis around the field before terminal punctuation too.\n    tail = tail.replace(\'**\',\'\').replace(\'__\',\'\').strip()\n    if not TAIL.fullmatch(tail):\n        return None, \'ambiguous_trailing_text\'\n    value = numeric(m[1])\n    return (value, \'parsed\') if value is not None else (None, \'unsupported_answer_value\')\n\ndef score(text, gold):\n    value, status = extract(text)\n    target = numeric(gold)\n    if target is None:\n        raise ValueError(\'Unsupported reference answer\')\n    m = FORMAT.fullmatch(text)\n    valid = bool(m and all(t.strip() for t in m.groups()) and all(text.count(t)==1 for t in (\'<reasoning>\',\'</reasoning>\',\'<answer>\',\'</answer>\')))\n    correct = value is not None and value == target\n    return dict(correct=correct, answer_parseable=value is not None,\n                extracted_answer=str(value) if value is not None else None,\n                extraction_status=status, format_valid=valid, structured_correct=correct and valid)\n')
Path('/content/test_eval_scoring.py').write_text("import unittest\nfrom eval_scoring import score\nclass PolicyTests(unittest.TestCase):\n def test_equivalent_presentations(self):\n  cases=[('<answer>25</answer>','25'),('<answer>25</answer>.','25'),('<answer>25 hours</answer>','25'),('<answer>25</answer> hours.','25'),('**<answer>25</answer>**','25'),('**<answer>25</answer>**.','25'),('<answer>$1,200.00</answer>','1200'),('<answer>-2.5</answer>','-2.5'),('<answer>20%</answer>','20'),('<answer>265 square feet</answer>','265'),('<answer>**25**</answer>','25'),('<answer>25.</answer>','25')]\n  for text,gold in cases:\n   with self.subTest(text=text):self.assertTrue(score(text,gold)['correct'])\n def test_rejections(self):\n  cases=['25','The answer is 25.','<answer>25','25</answer>','<answer>25</answer><answer>25</answer>','<answer>25</answer> or 26','<answer>25 or 26</answer>','<answer>2,5</answer>','<answer>NaN</answer>','<answer>2+23</answer>','<answer>25/1</answer>','<answer>2.5e1</answer>','<answer>25 unknownunit</answer>','<answer>25</answer> actually 26','</answer>25<answer>','<answer><answer>25</answer></answer>']\n  for text in cases:\n   with self.subTest(text=text):self.assertFalse(score(text,'25')['answer_parseable'])\n def test_wrong_remains_wrong(self):self.assertFalse(score('<answer>26 hours</answer>.','25')['correct'])\n def test_format_independent(self):\n  s=score('Work\\n<answer>25</answer>.','25');self.assertTrue(s['correct']);self.assertFalse(s['format_valid'])\n def test_full_format(self):self.assertTrue(score('<reasoning>work\\nwork</reasoning><answer>25</answer>','25')['structured_correct'])\n def test_never_gold_search(self):self.assertFalse(score('Intermediate value 25.\\n<answer>26</answer>','25')['correct'])\nif __name__=='__main__':unittest.main()\n")
Path('/content/evaluation_io.py').write_text('"""Durable evaluation logs; no GPU dependencies."""\nimport json,os\nfrom pathlib import Path\n\ndef read_rows(path, allowed_ids):\n    path=Path(path)\n    if not path.exists(): return {}\n    data=path.read_bytes(); lines=data.splitlines(keepends=True); rows={}; offset=0\n    for i,line in enumerate(lines):\n        try: row=json.loads(line)\n        except (ValueError, UnicodeDecodeError):\n            # Recover only an unfinished final append, preserving the original bytes.\n            if i!=len(lines)-1 or line.endswith(b\'\\n\'): raise\n            backup=path.with_name(path.name+\'.interrupted\')\n            if backup.exists(): raise RuntimeError(\'An interrupted log backup already exists; inspect it before recovery.\')\n            backup.write_bytes(data)\n            with path.open(\'r+b\') as f:f.truncate(offset)\n            break\n        key=(row[\'label\'],row[\'source_id\'])\n        if key[0] not in (\'base\',\'tuned\') or key[1] not in allowed_ids or key in rows:\n            raise ValueError(\'Unknown or duplicate evaluation row\')\n        rows[key]=row;offset+=len(line)\n    # A complete JSON object without its newline can also be a valid final write.\n    if rows and path.stat().st_size and not path.read_bytes().endswith(b\'\\n\'):\n        with path.open(\'ab\') as f:f.write(b\'\\n\');f.flush();os.fsync(f.fileno())\n    return rows\n\ndef append_row(path,row):\n    with Path(path).open(\'a\') as f:\n        f.write(json.dumps(row)+\'\\n\');f.flush();os.fsync(f.fileno())\n')
Path('/content/test_evaluation_io.py').write_text('import unittest,tempfile,json\nfrom pathlib import Path\nfrom evaluation_io import read_rows,append_row\nclass ResumeTests(unittest.TestCase):\n def setUp(self):\n  self.tmp=tempfile.TemporaryDirectory();self.addCleanup(self.tmp.cleanup);self.p=Path(self.tmp.name)/\'rows.jsonl\'\n def test_round_trip(self):\n  append_row(self.p,{\'label\':\'base\',\'source_id\':1});self.assertEqual(len(read_rows(self.p,{1})),1)\n def test_partial_recovery(self):\n  append_row(self.p,{\'label\':\'base\',\'source_id\':1})\n  with self.p.open(\'ab\') as f:f.write(b\'{"label":\')\n  self.assertEqual(len(read_rows(self.p,{1})),1);self.assertTrue(Path(str(self.p)+\'.interrupted\').exists())\n  append_row(self.p,{\'label\':\'tuned\',\'source_id\':1});self.assertEqual(len(read_rows(self.p,{1})),2)\n def test_duplicate_rejected(self):\n  for _ in range(2):append_row(self.p,{\'label\':\'base\',\'source_id\':1})\n  with self.assertRaises(ValueError):read_rows(self.p,{1})\n def test_unknown_rejected(self):\n  append_row(self.p,{\'label\':\'base\',\'source_id\':2})\n  with self.assertRaises(ValueError):read_rows(self.p,{1})\n def test_no_final_newline(self):\n  self.p.write_text(json.dumps({\'label\':\'base\',\'source_id\':1}));read_rows(self.p,{1})\n  self.assertTrue(self.p.read_bytes().endswith(b\'\\n\'))\nif __name__==\'__main__\':unittest.main()\n')
sys.path.insert(0,'/content')
subprocess.run([sys.executable,'-m','unittest','discover','-s','/content','-p','test_eval*.py','-v'],check=True)
from eval_scoring import score, VERSION
from evaluation_io import read_rows, append_row


## Frozen evaluation policy
Accept exactly one complete `<answer>…</answer>` field with a single decimal number. Normalize valid thousands separators, a leading currency symbol, terminal punctuation, Markdown emphasis, and an explicit whitelist of common unit labels (such as hours or square feet). Units are not converted. Reject duplicate/missing tags, multiple answers, arithmetic expressions and contradictory trailing prose. Never search the solution for the reference number.

`correct` is **tagged numeric-answer accuracy**, not a general assessment of every natural-language answer. Untagged outputs remain unparseable, even when a human can understand them. Report extraction coverage and strict full-format compliance separately. The policy is embedded in `eval_scoring.py` and hashed before generation. Freeze it for both arms.

Offline audit of the 80 prior training responses: **25 → 38** tagged answers accepted as correct; **55/80** parseable. This was a scorer-only change, not model improvement. Hypothetically rescoring these fixed groups gives reward variation in 13/20 groups instead of 14/20. It does not reconstruct how training under the new rewards would have proceeded. The pilot adapter was trained with the old v2 rewards.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
# Use the tokenizer saved alongside this exact adapter.
tokenizer=AutoTokenizer.from_pretrained(str(RUN/'adapter'))
tokenizer.padding_side='left'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
raw=load_dataset('openai/gsm8k','main',split='train')
raw=raw.add_column('source_id',list(range(len(raw))))
assert raw._fingerprint==manifest['fingerprint'], 'Dataset fingerprint changed; stop to verify data identity.'
split=raw.train_test_split(test_size=128,seed=CFG['seed'])
train_rows=[json.loads(l) for l in (RUN/'train.jsonl').read_text().splitlines() if l.strip()]
old_dev=[json.loads(l) for l in (RUN/'dev.jsonl').read_text().splitlines() if l.strip()]
train_ids=set(manifest['train_ids']); old_dev_ids=set(manifest['dev_ids'])
assert train_ids=={r['source_id'] for r in train_rows}
assert old_dev_ids=={r['source_id'] for r in old_dev}
assert train_ids.issubset(set(split['train']['source_id']))
assert old_dev_ids.issubset(set(split['test']['source_id']))
for r in train_rows+old_dev:
    assert raw[r['source_id']]['question']==r['question'], 'Saved question identity mismatch.'
questions=[]
for x in split['test']:
    if x['source_id'] in train_ids | old_dev_ids: continue
    prompt=[{'role':'system','content':manifest['system_prompt']},{'role':'user','content':x['question']}]
    if len(tokenizer.apply_chat_template(prompt,tokenize=True,add_generation_prompt=True))>CFG['max_prompt_length']: continue
    assert '####' in x['answer']
    questions.append(dict(source_id=x['source_id'],question=x['question'],prompt=prompt,gold=x['answer'].rsplit('####',1)[1].strip()))
questions=questions[:N_QUESTIONS]
assert len(questions)==N_QUESTIONS
assert {x['source_id'] for x in questions}.isdisjoint(train_ids | old_dev_ids)
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
versions={name:importlib.metadata.version(name) for name in ['torch','transformers','peft','bitsandbytes','datasets','accelerate']}
protocol=dict(model=CFG['model'],revision=revision['resolved_commit'],
    adapter_sha256=digest(RUN/'adapter/adapter_model.safetensors'),
    adapter_config_sha256=digest(RUN/'adapter/adapter_config.json'),
    tokenizer_sha256=digest(RUN/'adapter/tokenizer.json'),
    scorer_version=VERSION,scorer_sha256=digest('/content/eval_scoring.py'),
    questions=questions,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
    dataset_fingerprint=raw._fingerprint,versions=versions,
    gpu=torch.cuda.get_device_name(0),seed=CFG['seed'],
    purpose='Fresh development evaluation of old-v2-reward adapter using frozen v3 scoring')
protocol_text=json.dumps(protocol,sort_keys=True,indent=2)
protocol_hash=hashlib.sha256(protocol_text.encode()).hexdigest()
OUT=RUN/('evaluation_v3_'+protocol_hash[:12]);OUT.mkdir(exist_ok=True)
if (OUT/'protocol.json').exists():
    assert (OUT/'protocol.json').read_text()==protocol_text
else: (OUT/'protocol.json').write_text(protocol_text)
(OUT/'eval_scoring.py').write_text(Path('/content/eval_scoring.py').read_text())
print('Results folder:',OUT)
print('New development questions:',len(questions))
print('Progress already saved:',len(read_rows(OUT/'evaluation_predictions.jsonl',{q['source_id'] for q in questions})), '/ 64 answers')


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, set_seed
from peft import PeftModel, prepare_model_for_kbit_training
set_seed(CFG['seed'])
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
base=AutoModelForCausalLM.from_pretrained(CFG['model'],revision=revision['resolved_commit'],
    quantization_config=quant,device_map={'':0},dtype=torch.float16,attn_implementation='sdpa')
# Match the frozen non-quantized layer casting used in training; no optimizer or training.
base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=False)
model=PeftModel.from_pretrained(base,str(RUN/'adapter'),is_trainable=False)
model.eval()
model.config.use_cache=True
for parameter in model.parameters(): parameter.requires_grad_(False)
assert not any(p.requires_grad for p in model.parameters())
assert model.get_model_status().enabled is True, 'Adapter is not consistently enabled.'
generation=GenerationConfig(do_sample=False,max_new_tokens=MAX_NEW_TOKENS,
    pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,use_cache=True)
print('Base model and saved adapter loaded. All parameters are frozen.')


## Evaluate — rerun this cell if it says PAUSED
Each question receives one base response and one adapter response with identical settings. The two arms alternate order across questions. Answers already saved are skipped. The timer is checked between question pairs; model loading is outside the timer. Do not use these timings as a latency benchmark.


In [ ]:
from contextlib import nullcontext
allowed_ids={q['source_id'] for q in questions}
log_path=OUT/'evaluation_predictions.jsonl'
rows=read_rows(log_path,allowed_ids)
session_start=time.perf_counter()
for index,q in enumerate(questions):
    if all((label,q['source_id']) in rows for label in ['base','tuned']): continue
    if time.perf_counter()-session_start>=SESSION_BUDGET_SECONDS:
        print('PAUSED: rerun this same evaluation cell to finish remaining answers.');break
    text=tokenizer.apply_chat_template(q['prompt'],tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(text,return_tensors='pt',add_special_tokens=False).to('cuda')
    assert inputs['input_ids'].shape[1]<=CFG['max_prompt_length']
    order=['base','tuned'] if index%2==0 else ['tuned','base']
    for label in order:
        key=(label,q['source_id'])
        if key in rows:continue
        context=model.disable_adapter() if label=='base' else nullcontext()
        torch.cuda.synchronize();start=time.perf_counter()
        with context,torch.inference_mode():
            output=model.generate(**inputs,generation_config=generation)
        torch.cuda.synchronize();elapsed=time.perf_counter()-start
        ids=output[0,inputs['input_ids'].shape[1]:]
        response=tokenizer.decode(ids,skip_special_tokens=True)
        row=dict(label=label,source_id=q['source_id'],gold=q['gold'],response=response,
            generated_tokens=len(ids),hit_token_cap=len(ids)>=MAX_NEW_TOKENS,
            seconds=elapsed,protocol_hash=protocol_hash,**score(response,q['gold']))
        append_row(log_path,row);rows[key]=row
        print(f'{len(rows)}/{2*N_QUESTIONS} answers saved — {label}, question {index+1}',flush=True)
if len(rows)==2*N_QUESTIONS:print('COMPLETE: run the final summary cell.')


In [ ]:
import math,random
rows=read_rows(OUT/'evaluation_predictions.jsonl',{q['source_id'] for q in questions})
complete=len(rows)==2*N_QUESTIONS
summary=dict(complete=complete,expected_questions=N_QUESTIONS,saved_answers=len(rows),
    scorer_version=VERSION,protocol_hash=protocol_hash,
    note='Exploratory development comparison, not official GSM8K test performance. Training used old v2 rewards; evaluation uses frozen v3 scoring.')
if complete:
    pairs=[(rows[('base',q['source_id'])],rows[('tuned',q['source_id'])]) for q in questions]
    for label in ['base','tuned']:
        rs=[rows[(label,q['source_id'])] for q in questions]
        count=sum(r['correct'] for r in rs);n=len(rs);rate=count/n;z=1.96
        center=(rate+z*z/(2*n))/(1+z*z/n)
        radius=z*math.sqrt(rate*(1-rate)/n+z*z/(4*n*n))/(1+z*z/n)
        summary[label]=dict(n=n,correct=count,accuracy=rate,
            accuracy_wilson_95=[center-radius,center+radius],
            parseable=sum(r['answer_parseable'] for r in rs),
            format_valid=sum(r['format_valid'] for r in rs),
            hit_token_cap=sum(r['hit_token_cap'] for r in rs))
    diffs=[int(a['correct'])-int(b['correct']) for b,a in pairs]
    summary['changed_to_correct']=sum(d==1 for d in diffs)
    summary['changed_to_wrong']=sum(d==-1 for d in diffs)
    summary['accuracy_difference_percentage_points']=100*sum(diffs)/len(diffs)
    rng=random.Random(3407)
    boots=sorted(sum(rng.choices(diffs,k=len(diffs)))/len(diffs)*100 for _ in range(10000))
    summary['paired_bootstrap_95_difference_pp']=[boots[249],boots[9749]]
    summary['uncertainty_note']='Small-sample descriptive intervals; bootstrap can degenerate when no outcomes change. No proof of equivalence or general improvement.'
    summary['response_texts_changed']=sum(b['response']!=a['response'] for b,a in pairs)
    print('Finished. Send evaluation_summary.json and evaluation_predictions.jsonl from:',OUT)
else:
    print('Not complete. Rerun the evaluation cell before interpreting results.')
(OUT/'evaluation_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


## Preserve and interpret
This notebook never modifies the adapter or the old training logs. Preserve its protocol and outputs alongside the historical pilot. A higher offline score from a new parser is not a model improvement. The paired comparison here uses the same v3 parser for both base and tuned outputs and new development questions, but 32 examples still give limited precision.

Do not change the parser after seeing these results and then treat the changed score as independent confirmation. Record any subsequent policy changes separately. The final project should retain the old training reward definition and clearly distinguish it from this evaluation policy.

Implementation reference: [PEFT model loading and disabling adapters](https://huggingface.co/docs/peft/v0.17.0/en/package_reference/peft_model).